# ARKHER — nó de treino (Kaggle GPU)

Este notebook é um nó da rede de treinamento da ARKHER.
1. Ligue o acelerador: **Settings → Accelerator → GPU T4 x2** (ou P100).
2. Rode tudo. Se a sessão cair, rode de novo: o estado viaja no checkpoint.
3. Baixe o resultado em `/kaggle/working` e devolva ao repositório (ou use o sync privado).

In [ ]:
# 1) código do projeto + dependências
!git clone https://github.com/PlexztyRBXStudiosBR/ARKHERAI_resynced.git /kaggle/working/arkher
%cd /kaggle/working/arkher
!pip install -q torch pyyaml
import torch
print('GPU disponível:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 2) OPCIONAL: puxar um checkpoint maior do seu depósito privado
# Crie um repo PRIVADO no serviço de depósito e adicione dois Secrets no Kaggle
# (Add-ons → Secrets): ARKHER_SYNC_TOKEN e ARKHER_SYNC_REPO. Eles ficam
# disponíveis como variáveis de ambiente automaticamente. Depois descomente:
# !pip install -q huggingface_hub
# !PYTHONPATH=. python workers/sync/hf_sync.py pull
print('passo 2 ok')

In [ ]:
# 3) dataset: gerar, preparar e validar (licença/segredos/PII)
!PYTHONPATH=. python -m model.training.prepare_dataset
!PYTHONPATH=. python -m model.training.validate_dataset
!PYTHONPATH=. python -m model.training.train_tokenizer

In [ ]:
# 4) treinar — retoma do latest.pt se existir
import os
resume = '--resume model/checkpoints/latest.pt' if os.path.exists('model/checkpoints/latest.pt') else ''
!PYTHONPATH=. python -m model.training.train --epochs 6 --tag kaggle {resume}

In [ ]:
# 5) avaliar + relatório
!PYTHONPATH=. python -m model.training.evaluate
!PYTHONPATH=. python -m model.training.report

In [ ]:
# 6) resultado pronto para download na aba Output (latest.pt + relatório)
# Para devolver ao depósito privado: !PYTHONPATH=. python workers/sync/hf_sync.py push
from pathlib import Path
for f in ['model/checkpoints/latest.pt', 'model/checkpoints/training_state.json']:
    p = Path(f)
    print(f, p.stat().st_size if p.exists() else 'AUSENTE')